In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from replay.metrics import Recall, Precision, HitRate
import faiss
from functools import reduce
import datasets
import torch
from tqdm import tqdm
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

In [2]:
dataset = load_from_disk(f"{DATA_PATH}/user_events_20230501")
polars_ds = dataset.to_polars()

In [3]:
index = faiss.read_index("data/neural_index.faiss")

with open("data/item_ids", "rb") as fp:
    item_ids = pickle.load(fp)
    
item_embs = np.load("data/item_embeddings.npy")
itemid2idx = {item_id: idx for idx, item_id in enumerate(item_ids)}

In [19]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
    .filter(pl.col("event_id") == "item_view")
)

all_clicks = (
    train_interactions
    .select(
        pl.col("user_id"),
        pl.col("c2_name"),
        pl.col("name"),
        pl.col("item_id"),
        pl.col("stime"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
)

In [20]:
query_clicks_20 = (
    all_clicks
    .filter(pl.col("rn") <= 20)
    .select("item_id")
    .unique()
)

In [13]:
ids = query_clicks_20["item_id"].to_list()

batch_size = 4096
similar_items = {}

replace_func = np.vectorize(lambda x: item_ids[x])

for i in tqdm(range(0, len(ids), batch_size)):
    cur_ids = ids[i:i+batch_size]
    queries = item_embs[[itemid2idx[id_] for id_ in cur_ids]]
    _, idx = index.search(queries, k=20)
    recs = replace_func(idx)
    cur_similar_items = {cur_ids[i]: recs[i, :][recs[i, :] != cur_ids[i]].tolist() for i in range(len(cur_ids))}
    similar_items = {**similar_items, **cur_similar_items}

100%|██████████| 392/392 [48:23<00:00,  7.41s/it]


In [14]:
#with open("data/similar_items_neural_lst20_top20", "wb") as fp:
#    pickle.dump(similar_items, fp)

In [6]:
with open("data/similar_items_neural_lst20_top20", "rb") as fp:
    similar_items = pickle.load(fp)

In [21]:
def get_similar_items(row):
    return list(reduce(lambda x, y: x + y, [similar_items[item_id] for item_id in row["last_clicks"] if item_id in similar_items]))

In [22]:
user_last_clicks_20 = (
     all_clicks
    .filter(pl.col("rn") <= 20)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("last_clicks"))
)

In [23]:
recs_20 = (
    user_last_clicks_20
    .with_columns(
        pl.struct(["last_clicks"]).apply(get_similar_items).alias("recs")
    )
)

In [27]:
recs_stats = (
    recs_20
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
    .select(
        pl.min("recs_count").alias("min_recs_count"),
        pl.mean("recs_count").alias("mean_recs_count"),
        pl.max("recs_count").alias("max_recs_count")
    )
)

recs_stats

min_recs_count,mean_recs_count,max_recs_count
i64,f64,i64
19,193.280973,456


In [28]:
recs_20.rename({"recs": "neural_item2item_recs"}).select("user_id", "neural_item2item_recs").write_parquet("data/item2item_recs.parquet")

In [24]:
test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .filter(pl.col("event_id") == "item_view")
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        recs_20,
        on="user_id",
        how="inner"
    )
)

In [25]:
TOP_K_VALUES = [10, 100, 537]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def intersection(row):
    return len(set(row["recs"]) & set(row["future_clicks"]))

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [26]:
metrics

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.004806,0.023949,0.04927,0.004906,0.003072,0.001502,0.038641,0.171861,0.31711,4446.0,19774.0,36486.0
